# Pipeline de Ingestão da Tabela de Votação

Notebook responsável pela ingestão da tabela **votacao** utilizando PySpark e Delta Lake. O fluxo realiza a leitura dos arquivos CSV brutos, consolida os dados em um DataFrame e grava a camada Bronze no formato Delta.

#### 1. Importação das Bibliotecas

Nesta etapa importamos as bibliotecas necessárias para realizar a ingestão da tabela **consulta_candidatos**. Utilizamos o Spark para processamento distribuído e o Delta Lake para persistência dos dados na camada Bronze.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

print('Bibliotecas importadas com sucesso!')

Bibliotecas importadas com sucesso!


#### 2. Inicialização da Sessão Spark

Inicializamos a sessão Spark configurada para trabalhar com o Delta Lake. Essa sessão será utilizada durante todo o processo de ingestão da tabela de votação.

In [2]:
builder = (
    SparkSession.builder
        # .master('local[*]')
        .appName('TSE-Analytics-Validation-Analise-Votacao')
        .config(
            'spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension'
        )
        .config(
            'spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog'
        )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print('Sessão Spark criada com sucesso com suporte a Delta Lake!')
print(f'Versão do PySpark: {spark.version}')

:: loading settings :: url = jar:file:/usr/local/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/vscode/.ivy2.5.2/cache
The jars for the packages stored in: /home/vscode/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4be9d330-c2dd-4a3e-88cd-009ee4f8aba3;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.0 in central
	found io.delta#delta-storage;4.3.0 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.5.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.3.0 in

Sessão Spark criada com sucesso com suporte a Delta Lake!
Versão do PySpark: 4.1.1


#### 5. Leitura da Tabela Bronze e Criação da View Temporária

Nesta etapa, carregamos a tabela consulta_candidatos armazenada em formato Delta na camada Bronze para um DataFrame do PySpark. Em seguida, registramos esse DataFrame como uma view temporária chamada candidatos, permitindo a execução de consultas SQL durante as análises e validações do processo de ingestão.

In [ ]:
votacao = (spark.read
                .format('delta')
                .load('data/bronze/votacao'))

votacao.createOrReplaceTempView('votacao')

26/07/03 22:15:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [ ]:
spark.sql(""" 
    SELECT * FROM votacao
    """).show(10)


+----------+-------------------+-----------+---------------+-----------------+--------+----------+--------------------+----------+--------------+-----+-----+------+------------+------------+-------+--------+----------+------------+------------+--------------------+-----------------+-------------------+-----------------------+-----------------------+------------------------+------------------------+---------------+----------+----------+--------------------+------------+------------+------------+-----------------------+------------+--------------------+-----------------------+-------------------+-----------------+------------------------+-------------------------+----------------+----------------+
|DT_GERACAO|         HH_GERACAO|ANO_ELEICAO|CD_TIPO_ELEICAO|  NM_TIPO_ELEICAO|NR_TURNO|CD_ELEICAO|          DS_ELEICAO|DT_ELEICAO|TP_ABRANGENCIA|SG_UF|SG_UE| NM_UE|CD_MUNICIPIO|NM_MUNICIPIO|NR_ZONA|CD_CARGO|  DS_CARGO|SQ_CANDIDATO|NR_CANDIDATO|        NM_CANDIDATO|NM_URNA_CANDIDATO|NM_SOCIAL_CA